# Pitch-Shift Test-Time Augmentation for Basic Pitch

Cheap recall experiment: run Basic Pitch on the audio pitch-shifted by [-2,-1,0,+1,+2] semitones, shift detected notes back to concert pitch, aggregate across shifts with a vote threshold **k** (k=1 union/max-recall → k=5 intersection/max-precision).

**The real question (same as fine-tuning):** does TTA *expand the precision-recall frontier*, or just slide along it? We answer it by overlaying the TTA vote-k points on the baseline's full threshold-sweep curve. If a TTA point sits **above** the baseline curve (higher F50 than any single threshold achieves), TTA genuinely helps. If it lands on the curve, it's equivalent to just changing the threshold.

No training. Uses pretrained Basic Pitch only (no leakage concern). Default tests player 05; edit `TEST_PLAYERS` to expand.


## 1. Environment + Basic Pitch patches (wipe-safe)

In [1]:
!nvidia-smi -L
import os, subprocess
if not os.path.exists('/content/basic-pitch'):
    !git clone -q https://github.com/spotify/basic-pitch.git /content/basic-pitch
!pip install -q --no-deps -e /content/basic-pitch 2>/dev/null
!pip install -q librosa soundfile sox mirdata tensorflow mir_eval resampy==0.4.2

import sys; sys.path.insert(0,'/content/basic-pitch')
import warnings; warnings.filterwarnings('ignore')
import tensorflow as tf
print(f"TF: {tf.__version__}")

# Wipe-safe patch helper: restore from git if file looks broken, then apply replacements idempotently
def patch_file(relpath, repls, sentinel):
    path=f'/content/basic-pitch/basic_pitch/{relpath}'
    src=open(path).read()
    if sentinel not in src or len(src)<500:
        subprocess.run(['git','checkout',f'basic_pitch/{relpath}'], cwd='/content/basic-pitch')
        src=open(path).read()
    for find,rep in repls:
        if find in src: src=src.replace(find,rep)
    open(path,'w').write(src)

patch_file('layers/signal.py', [('rank = input_shape.rank','rank = len(input_shape)')], 'NormalizedLog')
patch_file('models.py', [
    ('x = tf.expand_dims(x, -1)\n    if use_batchnorm:','x = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x)\n    if use_batchnorm:'),
    ('x_contours_reduced = tf.expand_dims(x_contours, -1)','x_contours_reduced = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x_contours)'),
], 'def model(')
patch_file('nn.py', [('tf.debugging.assert_equal(tf.shape(x).shape, 4)','pass')], 'FlattenAudioCh')

import importlib, basic_pitch.layers.signal as _s, basic_pitch.nn as _n, basic_pitch.models as _m
importlib.reload(_s); importlib.reload(_n); importlib.reload(_m)
print("Patches applied (wipe-safe).")

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-fe24a5ec-e490-57d0-ec2b-f714deda28cb)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basic-pitch (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 134.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.8/263.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━

TF: 2.20.0
Patches applied (wipe-safe).


## 2. Drive + data

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import glob, shutil, json
import numpy as np
from pathlib import Path

DATA_ROOT=next((c for c in [Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
                            Path('/content/drive/MyDrive/FullGuitarSetData')] if (c/'JamsFiles').exists()),None)
if DATA_ROOT is None: raise FileNotFoundError("GuitarSet not found")
LOCAL_AUDIO,LOCAL_JAMS='/content/gs_audio','/content/gs_jams'
os.makedirs(LOCAL_AUDIO,exist_ok=True); os.makedirs(LOCAL_JAMS,exist_ok=True)
def cp(src,dst,ext):
    s=glob.glob(os.path.join(str(src),f'*.{ext}'))
    if len(glob.glob(os.path.join(dst,f'*.{ext}')))>=len(s): print(f"  {ext}: on SSD"); return
    print(f"  copying {len(s)} {ext}...")
    for f in s:
        try: shutil.copy2(f,dst)
        except shutil.SameFileError: pass
cp(DATA_ROOT/'AudioFiles',LOCAL_AUDIO,'wav'); cp(DATA_ROOT/'JamsFiles',LOCAL_JAMS,'jams')
print(f"Audio {len(glob.glob(LOCAL_AUDIO+'/*.wav'))} | JAMS {len(glob.glob(LOCAL_JAMS+'/*.jams'))}")

Mounted at /content/drive
  copying 360 wav...
  copying 360 jams...
Audio 360 | JAMS 360


## 3. Pretrained model + inference/eval helpers

In [3]:
import pandas as pd, librosa
from basic_pitch import ICASSP_2022_MODEL_PATH
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES, FFT_HOP, ANNOTATIONS_FPS
from basic_pitch.note_creation import model_output_to_notes

pretrained=tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
N_OL=30; OVERLAP_LEN=N_OL*FFT_HOP; HOP_SIZE=AUDIO_N_SAMPLES-OVERLAP_LEN
MIN_NOTE_LEN=int(np.round(58/1000*ANNOTATIONS_FPS))

def window_audio(a):
    a=np.concatenate([np.zeros(OVERLAP_LEN//2,np.float32),a.astype(np.float32)]); out=[]; st=0
    while st<len(a):
        w=a[st:st+AUDIO_N_SAMPLES]
        if len(w)<AUDIO_N_SAMPLES: w=np.pad(w,(0,AUDIO_N_SAMPLES-len(w)))
        out.append(w)
        if st+AUDIO_N_SAMPLES>=len(a): break
        st+=HOP_SIZE
    return np.stack(out)

def unwrap(st,orig):
    n=N_OL//2; tr=st[:,n:-n,:]; flat=tr.reshape(-1,tr.shape[-1])
    return flat[:int(np.floor(orig*(ANNOTATIONS_FPS/AUDIO_SAMPLE_RATE))),:]

def raw_outputs(y):
    x=tf.constant(window_audio(y)[...,None],tf.float32)
    out=pretrained.signatures['serving_default'](input_2=x)
    return {'onset':unwrap(out['onset'].numpy(),len(y)),'note':unwrap(out['note'].numpy(),len(y)),
            'contour':unwrap(out['contour'].numpy(),len(y))}

def decode(raw, ot, ft):
    _,ev=model_output_to_notes(raw, onset_thresh=ot, frame_thresh=ft, min_note_len=MIN_NOTE_LEN,
                               min_freq=None, max_freq=None, include_pitch_bends=False)
    return [(float(n[0]),float(n[1]),int(n[2])) for n in ev]  # (onset,offset,midi)

def load_gt(jp):
    jam=json.load(open(jp)); notes=[]
    for ann in jam.get('annotations',[]):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for o in ann['data']:
            notes.append({'onset':float(o['time']),'offset':float(o['time'])+float(o['duration']),'midi':int(round(float(o['value'])))})
    return sorted(notes,key=lambda n:n['onset'])

def match(gt,pred,tol=0.05):
    c=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p['midi'])==int(g['midi']) and abs(p['onset']-g['onset'])<=tol: c.append((abs(p['onset']-g['onset']),pi,gi))
    c.sort(); up,ug=set(),set()
    for _,pi,gi in c:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0; R=tp/(tp+fn) if tp+fn else 0
    return P,R,(2*P*R/(P+R) if P+R else 0)
print("Helpers ready.")

Helpers ready.


## 4. TTA: pitch-shift, shift-back, vote-aggregate

In [4]:
SHIFTS=[-2,-1,0,1,2]
PER_SHIFT_ONSET=0.40; PER_SHIFT_FRAME=0.30   # fixed detection threshold per shift

def tta_for_recording(audio_path):
    y,_=librosa.load(audio_path,sr=AUDIO_SAMPLE_RATE,mono=True)
    raw0=None; per_shift_notes={}
    for n in SHIFTS:
        ys = y if n==0 else librosa.effects.pitch_shift(y, sr=AUDIO_SAMPLE_RATE, n_steps=n)
        raw = raw_outputs(ys)
        if n==0: raw0=raw
        notes = decode(raw, PER_SHIFT_ONSET, PER_SHIFT_FRAME)
        per_shift_notes[n] = [(on,off,m-n) for (on,off,m) in notes]   # shift pitch back to concert
    return raw0, per_shift_notes

def aggregate(per_shift_notes, tol=0.05):
    items=[]
    for si,n in enumerate(SHIFTS):
        for (on,off,m) in per_shift_notes[n]:
            items.append([on,off,m,si])
    order=sorted(range(len(items)), key=lambda i: items[i][0])
    used=[False]*len(items); clusters=[]
    for i in order:
        if used[i]: continue
        on_i,off_i,m_i,si_i=items[i]; mem=[i]; used[i]=True; shifts={si_i}
        for j in order:
            if used[j]: continue
            on_j,off_j,m_j,si_j=items[j]
            if m_j==m_i and abs(on_j-on_i)<=tol and si_j not in shifts:
                mem.append(j); used[j]=True; shifts.add(si_j)
        ons=[items[k][0] for k in mem]; offs=[items[k][1] for k in mem]
        clusters.append({'onset':float(np.median(ons)),'offset':float(np.median(offs)),'midi':m_i,'votes':len(shifts)})
    return clusters

def votes_to_pred(clusters,k):
    return [{'onset':c['onset'],'offset':c['offset'],'midi':c['midi']} for c in clusters if c['votes']>=k]
print(f"TTA over shifts {SHIFTS}, per-shift threshold {PER_SHIFT_ONSET}/{PER_SHIFT_FRAME}")

TTA over shifts [-2, -1, 0, 1, 2], per-shift threshold 0.4/0.3


## 5. Run + compare TTA vs baseline frontier

In [5]:
TEST_PLAYERS=['05']     # <-- edit to ['00','02','05'] or all six for a fuller read

jams=[j for j in sorted(glob.glob(LOCAL_JAMS+'/*.jams'))
      if os.path.basename(j).split('_')[0] in TEST_PLAYERS]
print(f"Running TTA on {len(jams)} recordings (players {TEST_PLAYERS})... ~{len(jams)*5//30}+ min")

cache=[]
for i,jp in enumerate(jams):
    stem=os.path.splitext(os.path.basename(jp))[0]
    cands=(glob.glob(os.path.join(LOCAL_AUDIO,stem+'*mic*.wav')) or glob.glob(os.path.join(LOCAL_AUDIO,stem+'*.wav')))
    if not cands: continue
    raw0, psn = tta_for_recording(cands[0])
    cache.append({'raw0':raw0,'psn':psn,'gt':load_gt(jp)})
    if (i+1)%10==0: print(f"  {i+1}/{len(jams)}")
print("Inference done.")

def agg_eval(preds_per_rec, gts):
    rows=[match(g,p) for p,g in zip(preds_per_rec,gts)]
    ng=[len(g) for g in gts]
    P=np.average([r[0] for r in rows],weights=ng); R=np.average([r[1] for r in rows],weights=ng); F=np.average([r[2] for r in rows],weights=ng)
    return P,R,F

gts=[c['gt'] for c in cache]

# Baseline: threshold sweep on shift-0 raw outputs (traces the frontier)
print("\n--- BASELINE (Basic Pitch, threshold sweep) ---")
base_pts=[]
for ot in [0.10,0.15,0.20,0.25,0.30,0.40,0.50]:
    preds=[[{'onset':o,'offset':f,'midi':m} for (o,f,m) in decode(c['raw0'],ot,0.30)] for c in cache]
    P,R,F=agg_eval(preds,gts); base_pts.append((ot,P,R,F))
    print(f"  onset={ot:.2f}  P={P:.4f} R={R:.4f} F={F:.4f}")
base_best=max(base_pts,key=lambda t:t[3])

# TTA: vote-k sweep at fixed per-shift threshold
print("\n--- TTA (vote threshold k) ---")
clusters_per_rec=[aggregate(c['psn']) for c in cache]
tta_pts=[]
for k in [1,2,3,4,5]:
    preds=[votes_to_pred(cl,k) for cl in clusters_per_rec]
    P,R,F=agg_eval(preds,gts); tta_pts.append((k,P,R,F))
    print(f"  k>={k}  P={P:.4f} R={R:.4f} F={F:.4f}")
tta_best=max(tta_pts,key=lambda t:t[3])

print("\n"+"="*56)
print("VERDICT")
print("="*56)
print(f"Best baseline:  F50={base_best[3]:.4f}  (onset={base_best[0]}, P={base_best[1]:.3f} R={base_best[2]:.3f})")
print(f"Best TTA:       F50={tta_best[3]:.4f}  (k>={tta_best[0]}, P={tta_best[1]:.3f} R={tta_best[2]:.3f})")
print(f"TTA delta F50:  {tta_best[3]-base_best[3]:+.4f}")
print("-"*56)
if tta_best[3] > base_best[3] + 0.003:
    print("TTA EXPANDS the frontier — worth pursuing on the full player set.")
else:
    print("TTA lands ON the baseline frontier — equivalent to a threshold change. No free recall.")
print("="*56)
print(f"(players {TEST_PLAYERS}; expand TEST_PLAYERS for a fuller read)")

Running TTA on 60 recordings (players ['05'])... ~10+ min
  10/60
  20/60
  30/60
  40/60
  50/60
  60/60
Inference done.

--- BASELINE (Basic Pitch, threshold sweep) ---
  onset=0.10  P=0.2874 R=0.9092 F=0.4260
  onset=0.15  P=0.3612 R=0.9151 F=0.5058
  onset=0.20  P=0.4235 R=0.9166 F=0.5657
  onset=0.25  P=0.4834 R=0.9196 F=0.6190
  onset=0.30  P=0.5359 R=0.9192 F=0.6616
  onset=0.40  P=0.6336 R=0.9129 F=0.7327
  onset=0.50  P=0.6959 R=0.8989 F=0.7705

--- TTA (vote threshold k) ---
  k>=1  P=0.4456 R=0.9274 F=0.5893
  k>=2  P=0.5835 R=0.9066 F=0.6963
  k>=3  P=0.6903 R=0.8803 F=0.7608
  k>=4  P=0.7858 R=0.8241 F=0.7910
  k>=5  P=0.8589 R=0.6776 F=0.7434

VERDICT
Best baseline:  F50=0.7705  (onset=0.5, P=0.696 R=0.899)
Best TTA:       F50=0.7910  (k>=4, P=0.786 R=0.824)
TTA delta F50:  +0.0206
--------------------------------------------------------
TTA EXPANDS the frontier — worth pursuing on the full player set.
(players ['05']; expand TEST_PLAYERS for a fuller read)


In [6]:
print("--- EXTENDED BASELINE (high thresholds, same cached outputs) ---")
ext_pts=[]
for ot in [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]:
    preds=[[{'onset':o,'offset':f,'midi':m} for (o,f,m) in decode(c['raw0'],ot,0.30)] for c in cache]
    P,R,F=agg_eval(preds,gts); ext_pts.append((ot,P,R,F))
    print(f"  onset={ot:.2f}  P={P:.4f} R={R:.4f} F={F:.4f}")
ext_best=max(ext_pts,key=lambda t:t[3])
print(f"\nExtended baseline best: F50={ext_best[3]:.4f} (onset={ext_best[0]}, P={ext_best[1]:.3f} R={ext_best[2]:.3f})")
print(f"TTA best:               F50=0.7910 (k>=4, P=0.786 R=0.824)")
print(f"Honest delta vs FULL baseline sweep: {0.7910-ext_best[3]:+.4f}")

--- EXTENDED BASELINE (high thresholds, same cached outputs) ---
  onset=0.50  P=0.6959 R=0.8989 F=0.7705
  onset=0.55  P=0.7190 R=0.8905 F=0.7822
  onset=0.60  P=0.7383 R=0.8785 F=0.7892
  onset=0.65  P=0.7526 R=0.8623 F=0.7904
  onset=0.70  P=0.7636 R=0.8421 F=0.7866
  onset=0.75  P=0.7694 R=0.8138 F=0.7753
  onset=0.80  P=0.7697 R=0.7731 F=0.7532
  onset=0.85  P=0.7588 R=0.7138 F=0.7134
  onset=0.90  P=0.7217 R=0.6249 F=0.6440

Extended baseline best: F50=0.7904 (onset=0.65, P=0.753 R=0.862)
TTA best:               F50=0.7910 (k>=4, P=0.786 R=0.824)
Honest delta vs FULL baseline sweep: +0.0006
